# Unit 5 Exercise: Training your own Skipgram

Zen Anthony Pastolero &nbsp;|&nbsp; BSCS 3-A AI

---

### (10 points) Use a Wikipedia article as your dataset:

**Answer:** The Wikipedia URL assignment is on **Line 21** (`WIKI_URL = "https://en.wikipedia.org/wiki/Gundam"`). 
The actual fetching and extraction of the text from the HTML is handled by the `fetch_wikipedia_article(url: str)` function, 
located on **Lines 32–49**.


In [ ]:
import re
import math
import json
import random
from collections import Counter
from typing import List, Tuple, Dict

import requests
from bs4 import BeautifulSoup
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np


# The article should be reasonably long (at least a few thousand words) for good results.
WIKI_URL = "https://en.wikipedia.org/wiki/Gundam"
RANDOM_SEED = 42


def ensure_nltk():
    resources = ["punkt", "punkt_tab"]
    for r in resources:
        try:
            nltk.data.find(f"tokenizers/{r}")
        except LookupError:
            nltk.download(r)


def fetch_wikipedia_article(url: str) -> str:
    headers = {
        "User-Agent": "Mozilla/5.0 (compatible; SGNS-Gundam-Training/1.0)"
    }
    resp = requests.get(url, headers=headers, timeout=30)
    resp.raise_for_status()

    # Extract main content text from the Wikipedia page
    soup = BeautifulSoup(resp.text, "html.parser")

    content_div = soup.find("div", {"id": "mw-content-text"})
    if content_div is None:
        raise ValueError("Could not find Wikipedia article content.")

    paragraphs = content_div.find_all(["p", "li"])
    text_blocks = []

    for p in paragraphs:
        txt = p.get_text(" ", strip=True)
        if txt:
            text_blocks.append(txt)

    text = "\n".join(text_blocks)

    text = re.sub(r"\[[0-9]+\]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


---

### (10 points) Preprocess the text coming from the selected corpus.

**Answer:** The preprocessing steps (tokenization, lowercasing, regex cleaning) are handled within the 
`preprocess_text(text: str)` function on **Lines 52–79**.


In [ ]:
def preprocess_text(text: str) -> List[List[str]]:
    sentences = sent_tokenize(text)

    processed = []
    for sent in sentences:
        sent = sent.lower()
        sent = re.sub(r"[^a-z0-9\-\s]", " ", sent)
        sent = re.sub(r"\s+", " ", sent).strip()
        if not sent:
            continue

        tokens = word_tokenize(sent)

        cleaned = []
        for tok in tokens:
            tok = tok.strip("-")
            if not tok:
                continue
            if tok.isdigit():
                continue
            if len(tok) < 2:
                continue
            cleaned.append(tok)

        if len(cleaned) >= 3:
            processed.append(cleaned)

    return processed


def corpus_stats(sentences: List[List[str]]) -> Dict[str, int]:
    flat = [w for s in sentences for w in s]
    vocab = set(flat)
    return {
        "num_sentences": len(sentences),
        "num_tokens": len(flat),
        "vocab_size": len(vocab),
    }


---

### (10 points) Train a Skip-gram with Negative Sampling model.

**Answer:** The initialization and training of the model take place in the `train_sgns` function on **Lines 91–106**. 
The specific configuration for vector size (`vector_size=100`) and window (`window=5`) are found on 
**Lines 93 and 94**.



In [ ]:
def train_sgns(sentences: List[List[str]]) -> Word2Vec:
    model = Word2Vec(
        sentences=sentences,
        vector_size=100,  # What happens if we change this? Try 50, 200, 300 and see how it affects results.
        window=10,        # Updated from 5 to 10 as required by the assignment.
        min_count=1,
        workers=4,
        sg=1,             # 0 = CBOW, 1 = skip-gram
        negative=10,      # negative sampling
        epochs=200,
        sample=1e-3,
        alpha=0.025,
        min_alpha=0.0007,
        seed=RANDOM_SEED,
    )
    return model


---

### (10 points) Evaluate the embeddings using a small test set.

**Answer:** The evaluation lists are defined on **Lines 205–218** (`relatedness_test`) and **Lines 234–239** (`analogy_test`). 
The functions responsible are `evaluate_relatedness` (**Line 119**) and `evaluate_analogies` (**Line 136**).

---

### (10 points) Report nearest neighbors, similarity scores, and test-set performance.

**Answer:** The nearest neighbors are reported using `print_top_neighbors` (**Line 199**). 
Similarity scores and test-set performance are printed in the `main` function across **Lines 220–258**.


In [ ]:
def has_word(model: Word2Vec, word: str) -> bool:
    return word in model.wv.key_to_index


def cosine(model: Word2Vec, w1: str, w2: str) -> float:
    v1 = model.wv[w1].reshape(1, -1)
    v2 = model.wv[w2].reshape(1, -1)
    return float(cosine_similarity(v1, v2)[0][0])


def evaluate_relatedness(model: Word2Vec, test_pairs: List[Tuple[str, str, float]]):
    gold = []
    pred = []
    covered = []

    for w1, w2, score in test_pairs:
        if has_word(model, w1) and has_word(model, w2):
            sim = cosine(model, w1, w2)
            gold.append(score)
            pred.append(sim)
            covered.append((w1, w2, score, sim))

    return {
        "covered_items": covered,
        "coverage": len(covered),
        "total": len(test_pairs),
    }


def evaluate_analogies(model: Word2Vec, analogies: List[Tuple[str, str, str, str]]):
    """
    Analogy format: a:b :: c:d
    Checks whether most_similar(positive=[b,c], negative=[a]) returns d.
    """
    covered = 0
    correct = 0
    details = []

    for a, b, c, d in analogies:
        if all(has_word(model, w) for w in [a, b, c, d]):
            covered += 1
            try:
                preds = model.wv.most_similar(positive=[b, c], negative=[a], topn=5)
                predicted_words = [w for w, _ in preds]
                hit = d in predicted_words
                correct += int(hit)
                details.append({
                    "analogy": f"{a}:{b}::{c}:?",
                    "expected": d,
                    "predictions": predicted_words,
                    "correct_in_top5": hit
                })
            except KeyError:
                pass

    accuracy = correct / covered if covered else float("nan")
    return {
        "coverage": covered,
        "total": len(analogies),
        "accuracy_top5": accuracy,
        "details": details
    }


def print_top_neighbors(model: Word2Vec, words: List[str], topn: int = 8):
    print("\n=== Nearest Neighbors ===")
    for word in words:
        if has_word(model, word):
            neighbors = model.wv.most_similar(word, topn=topn)
            print(f"\n{word}:")
            for neigh, score in neighbors:
                print(f"  {neigh:20s} {score:.4f}")
        else:
            print(f"\n{word}: [OOV]")


---

### Answer the following questions:

---

#### (10 points) What are the critical parts of the script related on training the Word2Vec?

**Answer:** The initialization of the `Word2Vec()` object inside the `train_sgns` function is the most critical part. 
Specifically:
- Passing the tokenized corpus to `sentences` supplies the training data.
- Setting `sg=1` enforces the **Skip-gram** architecture (instead of CBOW, which would be `sg=0`).
- Setting `negative=10` activates **Negative Sampling** with 10 noise words per positive example.
- Defining `vector_size=100` controls the dimensionality of the output embeddings.
- Defining `window` controls the span of context words considered around each target word.

---

#### (10 points) What Word2Vec methods were used to evaluate the similarity of words?

**Answer:** The script primarily uses two methods:
1. `model.wv.most_similar()` — used to find the **nearest neighbors** of a given word and to **solve analogies** 
by passing `positive` and `negative` word lists.
2. `model.wv[word]` — used to extract the **raw numerical vector array** for a word, which is then passed 
to `sklearn.metrics.pairwise.cosine_similarity` inside the custom `cosine()` function to compute a direct similarity score.

---

### Update the code to:

#### (15 points) Retrain the Word2Vec model with a window size of 10. Compare the evaluation values to the previous model. What can you derive from the change?

The `train_sgns` function above has already been updated with `window=10`.

**Comparison — Window = 5 vs. Window = 10:**

| | Window = 5 (Original) | Window = 10 (Updated) |
|---|---|---|
| **Context span** | Immediate neighbors only | Broader sentence context |
| **Similarity type learned** | Syntactic / functional | Semantic / topical |
| **Effect** | Grammatically interchangeable words cluster tightly | Topically related words (even non-interchangeable) score higher |

**OLD (Window = 5):** Restricts context to immediately adjacent words. Embeddings prioritize **syntactic and functional similarities**; 
grammatically interchangeable words cluster tightly in the vector space.

**NEW (Window = 10):** The larger window captures **broader semantic and topical relationships**. 
Similarity scores for word pairs that are topically related but not syntactically interchangeable generally increase, 
because the model has seen them co-occurring across wider sentence spans during training.


---

### (15 points) Visualize the vectors using PCA. Generate at least 20 known words and feed it on the visualization.

The `plot_pca_embeddings` function below reduces the 100-dimensional word vectors to 2D using **Principal Component Analysis (PCA)** 
and plots 20 domain-relevant words from the Gundam Wikipedia article. This allows us to visually inspect whether semantically 
related words (e.g., `gundam`, `gunpla`, `robot`, `mecha`) cluster together in the embedding space.


In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA


def plot_pca_embeddings(model):
    words_to_plot = [
        "gundam", "mobile", "suit", "anime", "series",
        "film", "gunpla", "franchise", "robot", "war",
        "earth", "space", "colony", "battle", "weapon",
        "pilot", "character", "story", "japan", "bandai"
    ]
    valid_words = [w for w in words_to_plot if has_word(model, w)]
    if not valid_words:
        print("No valid words found in the vocabulary for PCA.")
        return

    word_vectors = np.array([model.wv[w] for w in valid_words])
    pca = PCA(n_components=2)
    pca_result = pca.fit_transform(word_vectors)

    plt.figure(figsize=(12, 8))
    plt.scatter(pca_result[:, 0], pca_result[:, 1], edgecolors="k", c="skyblue", s=100)
    for i, word in enumerate(valid_words):
        plt.annotate(word, xy=(pca_result[i, 0], pca_result[i, 1]), xytext=(5, 5),
                     textcoords="offset points", ha="right", va="bottom", fontsize=11)
    plt.title("PCA Visualization of Word2Vec Embeddings (Skip-gram, Window=10)")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.show()


def main():
    random.seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)
    ensure_nltk()

    print("Downloading Wikipedia article...")
    raw_text = fetch_wikipedia_article(WIKI_URL)

    print("Preprocessing text...")
    sentences = preprocess_text(raw_text)
    stats = corpus_stats(sentences)

    print("\n=== Corpus Stats ===")
    for k, v in stats.items():
        print(f"{k}: {v}")

    print("\nTraining Skip-gram with Negative Sampling...")
    model = train_sgns(sentences)

    print("\nVocabulary size learned:", len(model.wv))

    probe_words = [
        "gundam", "mobile", "suit", "anime", "series",
        "film", "gunpla", "franchise", "robot", "war"
    ]
    print_top_neighbors(model, probe_words, topn=8)

    # Domain-specific relatedness test set
    # Higher score means should be more semantically related
    relatedness_test = [
        ("gundam",    "gunpla",   0.95),
        ("gundam",    "mobile",   0.80),
        ("gundam",    "suit",     0.85),
        ("anime",     "series",   0.90),
        ("film",      "movie",    0.90),
        ("robot",     "mecha",    0.95),
        ("franchise", "series",   0.80),
        ("war",       "battle",   0.75),
        ("gundam",    "kitchen",  0.05),
        ("anime",     "tractor",  0.02),
        ("gunpla",    "movie",    0.25),
        ("robot",     "pilot",    0.45),
    ]

    rel_results = evaluate_relatedness(model, relatedness_test)

    print("\n=== Relatedness Test Set ===")
    print(f"Coverage: {rel_results['coverage']}/{rel_results['total']}")
    for w1, w2, gold, pred in rel_results["covered_items"]:
        print(f"{w1:10s} - {w2:10s} | gold={gold:.2f} pred={pred:.4f}")

    # Small analogy-style test set
    analogy_test = [
        ("movie",  "film",    "tv",     "series"),
        ("robot",  "mecha",   "war",    "battle"),
        ("anime",  "series",  "movie",  "film"),
        ("pilot",  "cockpit", "driver", "car"),
    ]

    analogy_results = evaluate_analogies(model, analogy_test)

    print("\n=== Analogy Test Set ===")
    print(f"Coverage: {analogy_results['coverage']}/{analogy_results['total']}")
    print(f"Top-5 accuracy: {analogy_results['accuracy_top5']}")
    for item in analogy_results["details"]:
        print(json.dumps(item, ensure_ascii=False))

    # Example direct similarity checks
    print("\n=== Direct Similarity Checks ===")
    check_pairs = [
        ("gundam", "gunpla"),
        ("gundam", "anime"),
        ("robot",  "mecha"),
        ("gundam", "kitchen"),
    ]
    for w1, w2 in check_pairs:
        if has_word(model, w1) and has_word(model, w2):
            print(f"{w1:10s} <-> {w2:10s}: {cosine(model, w1, w2):.4f}")
        else:
            print(f"{w1:10s} <-> {w2:10s}: OOV")

    # PCA Visualization
    plot_pca_embeddings(model)

    # Save model
    model.save("exercise_5_skipgram_sgns.model")
    print("\nSaved model to: exercise_5_skipgram_sgns.model")

    print("\nDone.")


main()
